In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 2 — IMPORT DAN PATH PROYEK
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import pandas as pd

ROOT_DIR = Path("/content/drive/MyDrive/eksperimen_baru")
BASELINE_V2_DIR = ROOT_DIR / "baseline_v2"

MANIFEST_DIR = (
    BASELINE_V2_DIR
    / "manifests"
)

METRICS_DIR = (
    BASELINE_V2_DIR
    / "metrics"
)

LOGS_DIR = (
    BASELINE_V2_DIR
    / "logs"
)

# Memastikan folder tersedia
for directory in [
    MANIFEST_DIR,
    METRICS_DIR,
    LOGS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )
print("Baseline V2 :", BASELINE_V2_DIR)
print("Manifest    :", MANIFEST_DIR)
print("Metrics     :", METRICS_DIR)
print("Logs        :", LOGS_DIR)

Baseline V2 : /content/drive/MyDrive/eksperimen_baru/baseline_v2
Manifest    : /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests
Metrics     : /content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics
Logs        : /content/drive/MyDrive/eksperimen_baru/baseline_v2/logs


In [ ]:
# ============================================================
# CELL 3 — PATH FILE SUMBER DAN OUTPUT
# ============================================================

CONFIG_PATH = (
    BASELINE_V2_DIR
    / "config"
    / "config.json"
)

# ------------------------------------------------------------
# Final grouped manifests
# ------------------------------------------------------------

DATASET_A_GROUPED_PATH = (
    MANIFEST_DIR
    / "dataset_a_grouped_manifest.csv"
)

DATASET_B_GROUPED_PATH = (
    MANIFEST_DIR
    / "dataset_b_grouped_manifest.csv"
)

COMBINED_GROUPED_PATH = (
    MANIFEST_DIR
    / "combined_grouped_manifest.csv"
)

# ------------------------------------------------------------
# Image integrity
# ------------------------------------------------------------

CORRUPT_IMAGES_PATH = (
    METRICS_DIR
    / "corrupt_images.csv"
)

# ------------------------------------------------------------
# Exact duplicate audit
# ------------------------------------------------------------

EXACT_DUPLICATES_PATH = (
    METRICS_DIR
    / "exact_duplicate_images.csv"
)

EXACT_REMOVALS_PATH = (
    METRICS_DIR
    / "exact_duplicate_removals.csv"
)

EXACT_CONFLICT_PATH = (
    METRICS_DIR
    / "exact_duplicate_label_conflicts.csv"
)

# ------------------------------------------------------------
# Perceptual / pHash audit
# ------------------------------------------------------------

PERCEPTUAL_PAIRS_PATH = (
    METRICS_DIR
    / "perceptual_duplicate_pairs.csv"
)

PERCEPTUAL_CANDIDATES_PATH = (
    METRICS_DIR
    / "perceptual_duplicate_candidates.csv"
)

PHASH_CONFLICT_PATH = (
    METRICS_DIR
    / "phash_label_conflicts.csv"
)

PHASH_CROSS_DATASET_PATH = (
    METRICS_DIR
    / "phash_cross_dataset_candidates.csv"
)

PHASH_RESOLUTION_PATH = (
    METRICS_DIR
    / "phash_cross_label_resolution.csv"
)

# ------------------------------------------------------------
# Group-aware validation
# ------------------------------------------------------------

GROUP_VALIDATION_PATH = (
    METRICS_DIR
    / "split_group_validation.csv"
)

# ------------------------------------------------------------
# Dataset B semantic validation
# ------------------------------------------------------------

MANUAL_LABEL_VALIDATION_PATH = (
    LOGS_DIR
    / "dataset_b_manual_label_validation.json"
)

# ------------------------------------------------------------
# Final outputs
# ------------------------------------------------------------

SIGNATURE_PATH = (
    MANIFEST_DIR
    / "dataset_manifest_signatures.json"
)

AUDIT_STATUS_PATH = (
    LOGS_DIR
    / "data_audit_status.json"
)

print("Path configuration selesai.")

Path configuration selesai.


In [ ]:
# ============================================================
# CELL 4 — CEK FILE SUMBER
# ============================================================

required_manifest_files = [
    DATASET_A_GROUPED_PATH,
    DATASET_B_GROUPED_PATH,
    COMBINED_GROUPED_PATH
]
missing_required_files = []

for file_path in required_manifest_files:
    exists = file_path.exists()
    print(
        f"{file_path.name:<42}: "
        f"{'ADA' if exists else 'TIDAK ADA'}"
    )

    if not exists:
        missing_required_files.append(
            str(file_path)
        )
if missing_required_files:
    raise FileNotFoundError(
        "Manifest final belum lengkap:\n"
        + "\n".join(
            missing_required_files
        )
    )
print("\nSemua manifest final tersedia.")

dataset_a_grouped_manifest.csv            : ADA
dataset_b_grouped_manifest.csv            : ADA
combined_grouped_manifest.csv             : ADA

Semua manifest final tersedia.


In [ ]:
# ============================================================
# CELL 5 — MEMBACA MANIFEST FINAL
# ============================================================

dataset_a_grouped_manifest = pd.read_csv(
    DATASET_A_GROUPED_PATH
)
dataset_b_grouped_manifest = pd.read_csv(
    DATASET_B_GROUPED_PATH
)
combined_grouped_manifest = pd.read_csv(
    COMBINED_GROUPED_PATH
)
print("Ukuran manifest:")
print(
    "Dataset A :",
    dataset_a_grouped_manifest.shape
)
print(
    "Dataset B :",
    dataset_b_grouped_manifest.shape
)
print("Gabungan  :", combined_grouped_manifest.shape)
required_columns = [
    "dataset",
    "relative_path",
    "label",
    "sha256",
    "split_group_id"
]

for manifest_name, dataframe in [
    (
        "Dataset A",
        dataset_a_grouped_manifest
    ),
    (
        "Dataset B",
        dataset_b_grouped_manifest
    ),
    (
        "Gabungan",
        combined_grouped_manifest
    )
]:
    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{manifest_name} kehilangan kolom: "
            f"{missing_columns}"
        )
print("\nValidasi kolom manifest: LULUS")

Ukuran manifest:
Dataset A : (3276, 34)
Dataset B : (240, 34)
Gabungan  : (3516, 34)

Validasi kolom manifest: LULUS


In [ ]:
# ============================================================
# CELL 6 — FUNGSI MANIFEST SIGNATURE
# ============================================================

def create_manifest_signature(
    dataframe: pd.DataFrame
) -> str:
    """
    Membuat sidik jari SHA-256 berdasarkan isi penting
    dari manifest final.
    """
    signature_columns = [
        "dataset",
        "relative_path",
        "label",
        "sha256",
        "split_group_id"
    ]

    signature_source = (
        dataframe[
            signature_columns
        ]
        .sort_values(
            [
                "dataset",
                "relative_path",
                "label"
            ]
        )
        .fillna("")
        .astype(str)
        .agg("|".join, axis=1)
        .str.cat(sep="\n")
    )

    signature = hashlib.sha256(
        signature_source.encode(
            "utf-8"
        )
    ).hexdigest()

    return signature
print("Fungsi manifest signature berhasil dibuat.")

Fungsi manifest signature berhasil dibuat.


In [ ]:
# ============================================================
# CELL 7 — MEMBUAT DATASET MANIFEST SIGNATURES
# ============================================================

dataset_signatures = {
    "signature_version": (
        "baseline_v2_grouped_manifest_v1"
    ),

    "dataset_a": {
        "row_count": int(
            len(
                dataset_a_grouped_manifest
            )
        ),
        "unique_sha256": int(
            dataset_a_grouped_manifest[
                "sha256"
            ].nunique(
                dropna=True
            )
        ),
        "unique_split_groups": int(
            dataset_a_grouped_manifest[
                "split_group_id"
            ].nunique(
                dropna=True
            )
        ),
        "manifest_signature": (
            create_manifest_signature(
                dataset_a_grouped_manifest
            )
        )
    },

    "dataset_b": {
        "row_count": int(
            len(
                dataset_b_grouped_manifest
            )
        ),
        "unique_sha256": int(
            dataset_b_grouped_manifest[
                "sha256"
            ].nunique(
                dropna=True
            )
        ),
        "unique_split_groups": int(
            dataset_b_grouped_manifest[
                "split_group_id"
            ].nunique(
                dropna=True
            )
        ),
        "manifest_signature": (
            create_manifest_signature(
                dataset_b_grouped_manifest
            )
        )
    },

    "combined": {
        "row_count": int(
            len(
                combined_grouped_manifest
            )
        ),
        "unique_sha256": int(
            combined_grouped_manifest[
                "sha256"
            ].nunique(
                dropna=True
            )
        ),
        "unique_split_groups": int(
            combined_grouped_manifest[
                "split_group_id"
            ].nunique(
                dropna=True
            )
        ),
        "manifest_signature": (
            create_manifest_signature(
                combined_grouped_manifest
            )
        )
    },

    "created_at": (
        datetime.now().isoformat()
    )
}
# Tidak membuat file duplikat.
with open(
    SIGNATURE_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        dataset_signatures,
        file,
        indent=4,
        ensure_ascii=False
    )
print(
    "Manifest signature berhasil dibuat:"
)

print(SIGNATURE_PATH)

print(
    "\nCombined signature:"
)

print(
    dataset_signatures[
        "combined"
    ][
        "manifest_signature"
    ]
)

Manifest signature berhasil dibuat:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_manifest_signatures.json

Combined signature:
c9b581947680f9276a699bfe100526911a436d0966beae918cb54522ef6b9d6a


In [ ]:
# ============================================================
# CELL 8 — REQUIRED CSV READER / FAIL-CLOSED
# ============================================================

def read_required_csv(
    file_path: Path,
    required_columns=None
) -> pd.DataFrame:

    if not file_path.exists():

        raise FileNotFoundError(
            "Required audit artifact tidak ditemukan:\n"
            f"{file_path}"
        )

    if file_path.stat().st_size == 0:

        raise RuntimeError(
            "Required audit artifact adalah zero-byte file:\n"
            f"{file_path}"
        )

    try:

        dataframe = pd.read_csv(
            file_path
        )

    except pd.errors.EmptyDataError as error:

        raise RuntimeError(
            "Required audit artifact tidak memiliki "
            "header yang dapat dibaca:\n"
            f"{file_path}"
        ) from error

    if required_columns is not None:

        missing_columns = (
            set(required_columns)
            -
            set(dataframe.columns)
        )

        if missing_columns:

            raise RuntimeError(
                f"{file_path.name} kehilangan kolom:\n"
                f"{sorted(missing_columns)}"
            )

    return dataframe


print(
    "Strict required-CSV reader berhasil dibuat."
)

Strict required-CSV reader berhasil dibuat.


In [ ]:
# ============================================================
# CELL 9 — MEMBACA HASIL AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. CONFIG
# ------------------------------------------------------------

if not CONFIG_PATH.exists():

    raise FileNotFoundError(
        "config.json tidak ditemukan:\n"
        f"{CONFIG_PATH}"
    )


with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    CONFIG = json.load(file)


print(
    "Config version:",
    CONFIG.get(
        "config_version",
        "UNKNOWN"
    )
)


# ------------------------------------------------------------
# 2. REQUIRED AUDIT ARTIFACTS
# ------------------------------------------------------------

corrupt_images = read_required_csv(
    CORRUPT_IMAGES_PATH
)


exact_duplicates = read_required_csv(
    EXACT_DUPLICATES_PATH,
    required_columns=[
        "sha256",
        "duplicate_scope"
    ]
)


exact_duplicate_removals = (
    read_required_csv(
        EXACT_REMOVALS_PATH
    )
)


exact_cross_label_conflicts = (
    read_required_csv(
        EXACT_CONFLICT_PATH
    )
)


perceptual_candidate_pairs = (
    read_required_csv(
        PERCEPTUAL_PAIRS_PATH,
        required_columns=[
            "review_group_id",
            "path_a",
            "path_b",
            "hamming_distance",
            "cross_dataset",
            "cross_label"
        ]
    )
)


perceptual_candidates = (
    read_required_csv(
        PERCEPTUAL_CANDIDATES_PATH,
        required_columns=[
            "path",
            "dataset",
            "label",
            "phash",
            "review_group_id",
            "near_duplicate_group_id"
        ]
    )
)


phash_cross_label_conflicts = (
    read_required_csv(
        PHASH_CONFLICT_PATH,
        required_columns=[
            "review_group_id",
            "path_a",
            "path_b",
            "hamming_distance",
            "cross_label"
        ]
    )
)


phash_cross_dataset_candidates = (
    read_required_csv(
        PHASH_CROSS_DATASET_PATH,
        required_columns=[
            "cross_dataset"
        ]
    )
)


phash_cross_label_resolution = (
    read_required_csv(
        PHASH_RESOLUTION_PATH,
        required_columns=[
            "review_group_id",
            "dataset",
            "label",
            "path",
            "relative_path",
            "phash",
            "resolution_action",
            "resolution_reason",
            "expert_reannotation"
        ]
    )
)


group_validation = (
    read_required_csv(
        GROUP_VALIDATION_PATH,
        required_columns=[
            "split_group_id",
            "dataset_count",
            "label_count",
            "path_count",
            "group_size"
        ]
    )
)


# ------------------------------------------------------------
# 3. SUMMARY
# ------------------------------------------------------------

print("\nRingkasan artifact audit:")

print(
    "Citra corrupt/error            :",
    len(corrupt_images)
)

print(
    "Baris exact duplicate          :",
    len(exact_duplicates)
)

print(
    "Exact duplicate dikeluarkan    :",
    len(exact_duplicate_removals)
)

print(
    "Konflik label exact duplicate  :",
    len(exact_cross_label_conflicts)
)

print(
    "Near-duplicate candidate pairs :",
    len(perceptual_candidate_pairs)
)

print(
    "Near-duplicate candidate images:",
    len(perceptual_candidates)
)

print(
    "pHash cross-label pairs        :",
    len(phash_cross_label_conflicts)
)

print(
    "pHash cross-dataset pairs      :",
    len(phash_cross_dataset_candidates)
)

print(
    "Cross-label resolution rows    :",
    len(phash_cross_label_resolution)
)

print(
    "Split groups tervalidasi       :",
    len(group_validation)
)

Config version: 2.1.0

Ringkasan artifact audit:
Citra corrupt/error            : 0
Baris exact duplicate          : 52
Exact duplicate dikeluarkan    : 26
Konflik label exact duplicate  : 0
Near-duplicate candidate pairs : 741
Near-duplicate candidate images: 814
pHash cross-label pairs        : 2
pHash cross-dataset pairs      : 0
Cross-label resolution rows    : 3
Split groups tervalidasi       : 2989


In [ ]:
# ============================================================
# CELL 10A — FINALISASI VALIDASI SEMANTIK LABEL DATASET B
# ============================================================

RAW_LABEL = "blight"

STANDARDIZED_LABEL = (
    "bacterial_leaf_blight"
)


# ------------------------------------------------------------
# 1. DATASET B SIZE
# ------------------------------------------------------------

if len(dataset_b_grouped_manifest) != 240:

    raise RuntimeError(
        "Jumlah Dataset B tidak sesuai.\n"
        f"Expected: 240\n"
        f"Found   : {len(dataset_b_grouped_manifest)}"
    )


# ------------------------------------------------------------
# 2. RECOVER RAW FOLDER LABEL
# ------------------------------------------------------------

dataset_b_semantic_check = (
    dataset_b_grouped_manifest
    .copy()
)


if (
    "raw_folder_label"
    in dataset_b_semantic_check.columns
):

    dataset_b_semantic_check[
        "_raw_folder_label"
    ] = (
        dataset_b_semantic_check[
            "raw_folder_label"
        ]
        .astype(str)
        .str.strip()
    )

elif (
    "path"
    in dataset_b_semantic_check.columns
):

    dataset_b_semantic_check[
        "_raw_folder_label"
    ] = (
        dataset_b_semantic_check[
            "path"
        ]
        .astype(str)
        .map(
            lambda value:
            Path(value).parent.name
        )
    )

else:

    raise RuntimeError(
        "Dataset B tidak memiliki "
        "raw_folder_label maupun path."
    )


# ------------------------------------------------------------
# 3. VALIDATE RAW `blight`
# ------------------------------------------------------------

blight_rows = (
    dataset_b_semantic_check[
        dataset_b_semantic_check[
            "_raw_folder_label"
        ] == RAW_LABEL
    ]
    .copy()
)


if len(blight_rows) != 80:

    raise RuntimeError(
        "Jumlah citra raw folder `blight` "
        "tidak sesuai.\n"
        f"Expected: 80\n"
        f"Found   : {len(blight_rows)}"
    )


mapped_labels = sorted(
    blight_rows[
        "label"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)


if mapped_labels != [
    STANDARDIZED_LABEL
]:

    raise RuntimeError(
        "Raw folder `blight` tidak dipetakan "
        "secara konsisten.\n"
        f"Found: {mapped_labels}"
    )


# ------------------------------------------------------------
# 4. FINAL CLASS DISTRIBUTION
# ------------------------------------------------------------

expected_class_counts = {
    "bacterial_leaf_blight": 80,
    "blast": 80,
    "tungro": 80
}


actual_class_counts = (
    dataset_b_grouped_manifest[
        "label"
    ]
    .astype(str)
    .value_counts()
    .to_dict()
)


if actual_class_counts != expected_class_counts:

    raise RuntimeError(
        "Distribusi Dataset B tidak sesuai.\n"
        f"Expected: {expected_class_counts}\n"
        f"Found   : {actual_class_counts}"
    )


# ------------------------------------------------------------
# 5. AUTHORITATIVE SEMANTIC VALIDATION
# ------------------------------------------------------------

manual_label_validation = {

    "raw_label":
        RAW_LABEL,

    "mapped_label":
        STANDARDIZED_LABEL,

    "accepted":
        True,

    "decision":
        "ACCEPT",

    "validation_status":
        "ACCEPTED",

    "validation_type":
        "literature_based_semantic_validation",

    "validation_basis": {

        "dataset_b_total_images":
            int(
                len(
                    dataset_b_grouped_manifest
                )
            ),

        "raw_blight_images":
            int(
                len(blight_rows)
            ),

        "final_class_distribution":
            {
                key: int(value)
                for key, value
                in actual_class_counts.items()
            },

        "manifest_mapping_consistent":
            True,

        "semantic_mapping":
            (
                "blight -> "
                "bacterial_leaf_blight"
            ),

        "expert_reannotation":
            False
    },

    "evidence": [
        {
            "source": (
                "Fawaiq et al. (2023), "
                "Rice Plant Disease Detection with "
                "Data Augmentation Using Transfer Learning"
            ),

            "doi": (
                "10.55248/"
                "gengpi.2023.4.4.35530"
            ),

            "dataset": (
                "Leaf Rice Disease Indonesia"
            ),

            "reported_classes": [
                "bacterial_leaf_blight",
                "blast",
                "tungro"
            ]
        }
    ],

    "review_note": (
        "The raw Dataset B folder label `blight` "
        "is standardized to `bacterial_leaf_blight`. "
        "This is a literature-supported semantic "
        "dataset-label mapping and not independent "
        "expert re-annotation of individual images."
    ),

    "validated_at":
        datetime.now().isoformat()
}


with open(
    MANUAL_LABEL_VALIDATION_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manual_label_validation,
        file,
        indent=4,
        ensure_ascii=False
    )


with open(
    MANUAL_LABEL_VALIDATION_PATH,
    "r",
    encoding="utf-8"
) as file:

    semantic_validation_reload = (
        json.load(file)
    )


if (
    semantic_validation_reload.get(
        "accepted"
    )
    is not True
):

    raise RuntimeError(
        "Semantic validation tidak tersimpan "
        "sebagai ACCEPTED."
    )


dataset_b_label_mapping_accepted = True


print(
    "Dataset B semantic validation: ACCEPTED"
)

print(
    "Mapping:",
    RAW_LABEL,
    "->",
    STANDARDIZED_LABEL
)

print(
    "Raw blight images:",
    len(blight_rows)
)

print(
    "Class distribution:",
    actual_class_counts
)

Dataset B semantic validation: ACCEPTED
Mapping: blight -> bacterial_leaf_blight
Raw blight images: 80
Class distribution: {'tungro': 80, 'bacterial_leaf_blight': 80, 'blast': 80}


In [ ]:
# ============================================================
# CELL 11 — MENGHITUNG KONDISI AUDIT DAN RESOLUSI
# ============================================================

# ------------------------------------------------------------
# 1. FINAL EXACT DUPLICATE CHECK
# ------------------------------------------------------------

remaining_exact_duplicate_rows = int(
    combined_grouped_manifest[
        "sha256"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)


cross_dataset_exact_rows = int(
    (
        exact_duplicates[
            "duplicate_scope"
        ]
        ==
        "cross_dataset"
    ).sum()
)


# ------------------------------------------------------------
# 2. RAW pHASH CROSS-LABEL DETECTION
# ------------------------------------------------------------

detected_cross_label_pair_count = int(
    len(
        phash_cross_label_conflicts
    )
)


detected_cross_label_group_ids = set(
    phash_cross_label_conflicts[
        "review_group_id"
    ]
    .dropna()
    .astype(str)
    .tolist()
)


# ------------------------------------------------------------
# 3. VALIDATE RESOLUTION ARTIFACT
# ------------------------------------------------------------

resolution = (
    phash_cross_label_resolution
    .copy()
)


resolution[
    "review_group_id"
] = (
    resolution[
        "review_group_id"
    ]
    .astype(str)
)


resolution[
    "path"
] = (
    resolution[
        "path"
    ]
    .astype(str)
)


invalid_resolution_actions = (
    resolution[
        ~resolution[
            "resolution_action"
        ]
        .eq(
            "exclude_from_modeling"
        )
    ]
)


if not invalid_resolution_actions.empty:

    raise RuntimeError(
        "Ditemukan resolution_action yang "
        "tidak sesuai dengan protokol."
    )


# Tidak boleh mengklaim expert reannotation.

expert_reannotation_true = (
    resolution[
        "expert_reannotation"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            "true",
            "1",
            "yes"
        ]
    )
)


if expert_reannotation_true.any():

    raise RuntimeError(
        "Resolution artifact tidak boleh "
        "mengklaim expert re-annotation."
    )


resolution_group_ids = set(
    resolution[
        "review_group_id"
    ]
    .dropna()
    .astype(str)
    .tolist()
)


extra_resolution_groups = (
    resolution_group_ids
    -
    detected_cross_label_group_ids
)


if extra_resolution_groups:

    raise RuntimeError(
        "Resolution artifact mengandung "
        "review group yang bukan cross-label conflict:\n"
        f"{sorted(extra_resolution_groups)}"
    )


# ------------------------------------------------------------
# 4. SELURUH CONFLICT COMPONENT HARUS DIRESOLUSI
# ------------------------------------------------------------

conflict_component_rows = (
    perceptual_candidates[
        perceptual_candidates[
            "review_group_id"
        ]
        .astype(str)
        .isin(
            detected_cross_label_group_ids
        )
    ]
    .copy()
)


expected_resolution_paths = set(
    conflict_component_rows[
        "path"
    ]
    .astype(str)
    .tolist()
)


actual_resolution_paths = set(
    resolution[
        resolution[
            "review_group_id"
        ]
        .isin(
            detected_cross_label_group_ids
        )
    ][
        "path"
    ]
    .astype(str)
    .tolist()
)


if (
    actual_resolution_paths
    !=
    expected_resolution_paths
):

    raise RuntimeError(
        "Resolution artifact tidak mencakup "
        "seluruh cross-label conflict component.\n"
        f"Expected paths: {len(expected_resolution_paths)}\n"
        f"Resolved paths: {len(actual_resolution_paths)}"
    )


# ------------------------------------------------------------
# 5. RESOLVED PATHS TIDAK BOLEH ADA DI FINAL MANIFEST
# ------------------------------------------------------------

final_manifest_paths = set(
    combined_grouped_manifest[
        "path"
    ]
    .astype(str)
    .tolist()
)


resolved_paths_still_present = (
    actual_resolution_paths
    &
    final_manifest_paths
)


if resolved_paths_still_present:

    raise RuntimeError(
        "Citra cross-label yang sudah di-resolve "
        "masih berada di final manifest:\n"
        +
        "\n".join(
            sorted(
                resolved_paths_still_present
            )
        )
    )


# ------------------------------------------------------------
# 6. DETECT RESOLVED / UNRESOLVED GROUPS
# ------------------------------------------------------------

resolved_cross_label_group_ids = (
    detected_cross_label_group_ids
    &
    resolution_group_ids
)


unresolved_cross_label_group_ids = (
    detected_cross_label_group_ids
    -
    resolved_cross_label_group_ids
)


unresolved_cross_label_pair_count = int(
    phash_cross_label_conflicts[
        phash_cross_label_conflicts[
            "review_group_id"
        ]
        .astype(str)
        .isin(
            unresolved_cross_label_group_ids
        )
    ]
    .shape[0]
)


resolved_cross_label_pair_count = int(
    detected_cross_label_pair_count
    -
    unresolved_cross_label_pair_count
)


resolved_cross_label_image_count = int(
    len(
        actual_resolution_paths
    )
)


# ------------------------------------------------------------
# 7. pHASH CROSS-DATASET
# ------------------------------------------------------------

cross_dataset_phash_pair_count = int(
    len(
        phash_cross_dataset_candidates
    )
)


# ------------------------------------------------------------
# 8. GROUP-AWARE VALIDATION
# ------------------------------------------------------------

cross_dataset_group_count = int(
    (
        group_validation[
            "dataset_count"
        ] > 1
    ).sum()
)


cross_label_group_count = int(
    (
        group_validation[
            "label_count"
        ] > 1
    ).sum()
)


duplicate_path_group_count = int(
    (
        group_validation[
            "path_count"
        ]
        !=
        group_validation[
            "group_size"
        ]
    ).sum()
)


empty_split_group_count = int(
    (
        combined_grouped_manifest[
            "split_group_id"
        ]
        .isna()
        |
        combined_grouped_manifest[
            "split_group_id"
        ]
        .astype(str)
        .str.strip()
        .eq("")
    ).sum()
)


# ------------------------------------------------------------
# 9. CANDIDATE COUNTS
# ------------------------------------------------------------

perceptual_candidate_groups = int(
    perceptual_candidates[
        "review_group_id"
    ]
    .nunique(
        dropna=True
    )
)


if (
    "is_perceptual_candidate"
    not in combined_grouped_manifest.columns
):

    raise RuntimeError(
        "Final manifest kehilangan "
        "is_perceptual_candidate."
    )


retained_perceptual_candidate_images = int(
    combined_grouped_manifest[
        "is_perceptual_candidate"
    ]
    .sum()
)


# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------

print(
    "Exact duplicate tersisa        :",
    remaining_exact_duplicate_rows
)

print(
    "Exact lintas dataset terdeteksi:",
    cross_dataset_exact_rows
)

print(
    "\npHash cross-label pairs detected:",
    detected_cross_label_pair_count
)

print(
    "pHash cross-label pairs resolved:",
    resolved_cross_label_pair_count
)

print(
    "pHash cross-label pairs unresolved:",
    unresolved_cross_label_pair_count
)

print(
    "Cross-label groups detected    :",
    len(
        detected_cross_label_group_ids
    )
)

print(
    "Cross-label groups resolved    :",
    len(
        resolved_cross_label_group_ids
    )
)

print(
    "Cross-label images excluded    :",
    resolved_cross_label_image_count
)

print(
    "pHash cross-dataset pairs      :",
    cross_dataset_phash_pair_count
)

print(
    "\nVisual-review groups detected  :",
    perceptual_candidate_groups
)

print(
    "Candidate images retained      :",
    retained_perceptual_candidate_images
)

print(
    "Group lintas dataset           :",
    cross_dataset_group_count
)

print(
    "Group lintas label             :",
    cross_label_group_count
)

print(
    "Group dengan path duplikat     :",
    duplicate_path_group_count
)

print(
    "Split group ID kosong          :",
    empty_split_group_count
)

Exact duplicate tersisa        : 0
Exact lintas dataset terdeteksi: 0

pHash cross-label pairs detected: 2
pHash cross-label pairs resolved: 2
pHash cross-label pairs unresolved: 0
Cross-label groups detected    : 1
Cross-label groups resolved    : 1
Cross-label images excluded    : 3
pHash cross-dataset pairs      : 0

Visual-review groups detected  : 285
Candidate images retained      : 811
Group lintas dataset           : 0
Group lintas label             : 0
Group dengan path duplikat     : 0
Split group ID kosong          : 0


In [ ]:
# ============================================================
# CELL 12 — STATUS AUDIT FINAL
# ============================================================

audit_policy = (
    CONFIG.get(
        "data_audit",
        {}
    )
)


cross_label_phash_is_blocking = bool(
    audit_policy.get(
        "cross_label_near_duplicate_is_blocking",
        True
    )
)


cross_dataset_phash_is_blocking = bool(
    audit_policy.get(
        "cross_dataset_near_duplicate_is_blocking",
        True
    )
)


# ------------------------------------------------------------
# BLOCKING = UNRESOLVED ISSUES ONLY
# ------------------------------------------------------------

blocking_components = {

    "corrupt_images":
        int(
            len(corrupt_images)
        ),

    "exact_cross_label_conflicts":
        int(
            len(
                exact_cross_label_conflicts
            )
        ),

    "remaining_exact_duplicates":
        int(
            remaining_exact_duplicate_rows
        ),

    "cross_dataset_groups":
        int(
            cross_dataset_group_count
        ),

    "cross_label_groups":
        int(
            cross_label_group_count
        ),

    "duplicate_path_groups":
        int(
            duplicate_path_group_count
        ),

    "empty_split_group_ids":
        int(
            empty_split_group_count
        ),

    "unresolved_phash_cross_label_pairs":
        (
            int(
                unresolved_cross_label_pair_count
            )
            if
            cross_label_phash_is_blocking
            else
            0
        ),

    "phash_cross_dataset_pairs":
        (
            int(
                cross_dataset_phash_pair_count
            )
            if
            cross_dataset_phash_is_blocking
            else
            0
        )
}


blocking_issue_count = int(
    sum(
        blocking_components.values()
    )
)


if not dataset_b_label_mapping_accepted:

    audit_status = (
        "PENDING_SEMANTIC_LABEL_VALIDATION"
    )

elif blocking_issue_count > 0:

    audit_status = "BLOCKED"

else:

    audit_status = "PASSED"


print(
    "Cross-label pHash blocking :",
    cross_label_phash_is_blocking
)

print(
    "Cross-dataset pHash blocking:",
    cross_dataset_phash_is_blocking
)


print("\nBlocking components:")

for issue_name, issue_count in (
    blocking_components.items()
):

    print(
        f"- {issue_name:<38}: "
        f"{issue_count}"
    )


print(
    "\nBlocking issue count:",
    blocking_issue_count
)

print(
    "Status audit:",
    audit_status
)

Cross-label pHash blocking : True
Cross-dataset pHash blocking: True

Blocking components:
- corrupt_images                        : 0
- exact_cross_label_conflicts           : 0
- remaining_exact_duplicates            : 0
- cross_dataset_groups                  : 0
- cross_label_groups                    : 0
- duplicate_path_groups                 : 0
- empty_split_group_ids                 : 0
- unresolved_phash_cross_label_pairs    : 0
- phash_cross_dataset_pairs             : 0

Blocking issue count: 0
Status audit: PASSED


In [ ]:
# ============================================================
# CELL 13 — MENYIMPAN DATA AUDIT STATUS
# ============================================================

audit_status_record = {

    "audit_status":
        audit_status,

    "dataset_counts": {

        "dataset_a_final_rows":
            int(
                len(
                    dataset_a_grouped_manifest
                )
            ),

        "dataset_b_final_rows":
            int(
                len(
                    dataset_b_grouped_manifest
                )
            ),

        "combined_final_rows":
            int(
                len(
                    combined_grouped_manifest
                )
            )
    },


    "image_integrity": {

        "corrupt_or_error_rows":
            int(
                len(corrupt_images)
            )
    },


    "exact_duplicate_audit": {

        "duplicate_rows_detected":
            int(
                len(exact_duplicates)
            ),

        "duplicate_rows_removed":
            int(
                len(
                    exact_duplicate_removals
                )
            ),

        "remaining_duplicate_rows":
            int(
                remaining_exact_duplicate_rows
            ),

        "cross_label_conflict_rows":
            int(
                len(
                    exact_cross_label_conflicts
                )
            ),

        "cross_dataset_detected_rows":
            int(
                cross_dataset_exact_rows
            )
    },


    "perceptual_duplicate_audit": {

        "candidate_pairs_detected":
            int(
                len(
                    perceptual_candidate_pairs
                )
            ),

        "candidate_images_detected":
            int(
                len(
                    perceptual_candidates
                )
            ),

        "visual_review_groups":
            int(
                perceptual_candidate_groups
            ),

        "cross_label_pairs_detected":
            int(
                detected_cross_label_pair_count
            ),

        "cross_label_pairs_resolved":
            int(
                resolved_cross_label_pair_count
            ),

        "cross_label_pairs_unresolved":
            int(
                unresolved_cross_label_pair_count
            ),

        "cross_label_groups_detected":
            int(
                len(
                    detected_cross_label_group_ids
                )
            ),

        "cross_label_groups_resolved":
            int(
                len(
                    resolved_cross_label_group_ids
                )
            ),

        "cross_label_images_excluded":
            int(
                resolved_cross_label_image_count
            ),

        "cross_dataset_candidate_pairs":
            int(
                cross_dataset_phash_pair_count
            ),

        "candidate_images_retained":
            int(
                retained_perceptual_candidate_images
            ),

        "hamming_threshold":
            int(
                CONFIG[
                    "data_audit"
                ][
                    "phash_hamming_threshold"
                ]
            ),

        "resolution_strategy": (
            "Ambiguous cross-label near-duplicate "
            "connected components are conservatively "
            "excluded from modeling without relabeling."
        ),

        "resolution_artifact":
            str(
                PHASH_RESOLUTION_PATH
            ),

        "expert_reannotation":
            False
    },


    "group_validation": {

        "cross_dataset_groups":
            int(
                cross_dataset_group_count
            ),

        "cross_label_groups":
            int(
                cross_label_group_count
            ),

        "duplicate_path_groups":
            int(
                duplicate_path_group_count
            ),

        "empty_split_group_ids":
            int(
                empty_split_group_count
            )
    },


    "semantic_label_validation": {

        "dataset":
            "Dataset_B",

        "raw_label":
            RAW_LABEL,

        "mapped_label":
            STANDARDIZED_LABEL,

        "accepted":
            bool(
                dataset_b_label_mapping_accepted
            ),

        "validation_type":
            "literature_based_semantic_validation",

        "expert_reannotation":
            False
    },


    "blocking_issues":
        blocking_components,

    "blocking_issue_count":
        int(
            blocking_issue_count
        ),


    "dataset_signatures": {

        "dataset_a":
            dataset_signatures[
                "dataset_a"
            ][
                "manifest_signature"
            ],

        "dataset_b":
            dataset_signatures[
                "dataset_b"
            ][
                "manifest_signature"
            ],

        "combined":
            dataset_signatures[
                "combined"
            ][
                "manifest_signature"
            ]
    },


    "created_at":
        datetime.now().isoformat()
}


with open(
    AUDIT_STATUS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        audit_status_record,
        file,
        indent=4,
        ensure_ascii=False
    )


with open(
    AUDIT_STATUS_PATH,
    "r",
    encoding="utf-8"
) as file:

    audit_status_reload = (
        json.load(file)
    )


if (
    audit_status_reload.get(
        "audit_status"
    )
    !=
    audit_status
):

    raise RuntimeError(
        "Audit status tersimpan tidak "
        "sesuai dengan status di memory."
    )


print(
    "Data audit status berhasil disimpan:"
)

print(
    AUDIT_STATUS_PATH
)

print(
    "\nStatus:",
    audit_status
)

print(
    "Blocking issues:",
    blocking_issue_count
)

Data audit status berhasil disimpan:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/logs/data_audit_status.json

Status: PASSED
Blocking issues: 0


In [ ]:
# ============================================================
# CELL 14 — FINAL DATA AUDIT GATE
# ============================================================

final_files = [
    SIGNATURE_PATH,
    AUDIT_STATUS_PATH,
    MANUAL_LABEL_VALIDATION_PATH,
    PHASH_RESOLUTION_PATH
]


missing_files = [
    str(file_path)
    for file_path
    in final_files
    if not file_path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Final audit artifact belum lengkap:\n\n"
        +
        "\n".join(
            missing_files
        )
    )


# ------------------------------------------------------------
# VERIFY SEMANTIC VALIDATION
# ------------------------------------------------------------

with open(
    MANUAL_LABEL_VALIDATION_PATH,
    "r",
    encoding="utf-8"
) as file:

    final_semantic_validation = (
        json.load(file)
    )


if (
    final_semantic_validation.get(
        "accepted"
    )
    is not True
):

    raise RuntimeError(
        "Dataset B semantic mapping "
        "belum ACCEPTED."
    )


# ------------------------------------------------------------
# VERIFY AUDIT STATUS
# ------------------------------------------------------------

with open(
    AUDIT_STATUS_PATH,
    "r",
    encoding="utf-8"
) as file:

    final_audit_record = (
        json.load(file)
    )


if (
    final_audit_record.get(
        "audit_status"
    )
    !=
    audit_status
):

    raise RuntimeError(
        "Audit status file dan memory "
        "tidak konsisten."
    )


# ------------------------------------------------------------
# FINAL GATE
# ------------------------------------------------------------

if audit_status != "PASSED":

    print(
        "\n" + "=" * 72
    )

    print(
        "FINAL DATA AUDIT: NOT YET PASSED"
    )

    print(
        "=" * 72
    )

    print(
        "Current status :",
        audit_status
    )

    print(
        "Blocking issue :",
        blocking_issue_count
    )


    for issue_name, issue_count in (
        blocking_components.items()
    ):

        if issue_count > 0:

            print(
                f"- {issue_name}: "
                f"{issue_count}"
            )


    raise RuntimeError(
        "Final audit belum dapat dikunci."
    )


if blocking_issue_count != 0:

    raise RuntimeError(
        "Status PASSED tetapi masih "
        "terdapat blocking issue."
    )


if unresolved_cross_label_pair_count != 0:

    raise RuntimeError(
        "Masih terdapat unresolved "
        "cross-label pHash pairs."
    )


# ------------------------------------------------------------
# FINAL SUCCESS
# ------------------------------------------------------------

print(
    "\n" + "=" * 72
)

print(
    "FINAL DATA AUDIT: PASSED"
)

print(
    "=" * 72
)

print(
    "Dataset A rows                :",
    len(
        dataset_a_grouped_manifest
    )
)

print(
    "Dataset B rows                :",
    len(
        dataset_b_grouped_manifest
    )
)

print(
    "Combined rows                 :",
    len(
        combined_grouped_manifest
    )
)

print(
    "Cross-label pairs detected    :",
    detected_cross_label_pair_count
)

print(
    "Cross-label pairs resolved    :",
    resolved_cross_label_pair_count
)

print(
    "Cross-label pairs unresolved  :",
    unresolved_cross_label_pair_count
)

print(
    "Cross-label images excluded   :",
    resolved_cross_label_image_count
)

print(
    "Cross-dataset pHash pairs     :",
    cross_dataset_phash_pair_count
)

print(
    "Blocking issue                :",
    blocking_issue_count
)

print(
    "Dataset B label mapping       : ACCEPTED"
)

print(
    "=" * 72
)


FINAL DATA AUDIT: PASSED
Dataset A rows                : 3276
Dataset B rows                : 240
Combined rows                 : 3516
Cross-label pairs detected    : 2
Cross-label pairs resolved    : 2
Cross-label pairs unresolved  : 0
Cross-label images excluded   : 3
Cross-dataset pHash pairs     : 0
Blocking issue                : 0
Dataset B label mapping       : ACCEPTED
